# Capstone — CTR / Engagement Opportunity Scoring

This notebook mirrors the deployed research paper. It summarizes the completed Weeks 1–7 work using public-safe aggregate results. It deliberately does **not** include client-level examples, private queries, raw URLs, or credentials.

> Claim language: observed, measured, directional, decision-support.


## 1. Question

**Research question:** Among search-visible content items with enough exposure to matter, which items appear to under-capture clicks relative to their position context, and can a learned ranking improve the ordering of a limited human review queue?

**Decision:** which pages deserve review first.

**Unit:** one client × content item after aggregating March 2026 daily performance.


In [ ]:
# Public-safe headline facts from the completed internship notebooks.
facts = {
    "warehouse_daily_rows": 78_835_655,
    "march_fact_rows": 9_841_378,
    "visible_client_content_rows": 91_974,
    "train_rows": 85_429,
    "test_rows": 6_545,
    "train_clients": 31,
    "test_clients": 11,
    "client_overlap": 0,
}
facts


## 2. Data

**Release:** `flyrank_pseudonymized_warehouse_release_v20260703`

**Primary tables:** `fact_content_daily_performance` and `dim_content`; identifiers are used only for grouping/joining.

**Feature window:** March 1–31, 2026.

**Population:** GSC-available observations, aggregated to client × content, with at least 150 impressions and valid average position.

**Excluded:** client/content identifiers from model features; CTR and CTR-gap from the final model feature set; product flags; future/private fields; row-level client examples from the public artifact.

The release contains 78,835,655 daily fact rows. The March slice used for this lane contains 9,841,378 fact rows before aggregation.


## 3. Methodology

**Proxy label:** March `ctr_gap` = observed CTR minus the median CTR for the page's position tier. Positive class = worst 10% of the `ctr_gap` distribution. This is a current-window screening proxy, not a future causal outcome.

**Features:** `position_tier`, `word_count`, `content_type`, `main_intent`, `search_volume`, `impressions`.

**Baseline:** Week-4 transparent position-aware rule, evaluated unchanged.

**Models:** logistic regression and random forest.

**Validation:** `GroupShuffleSplit(test_size=0.25, random_state=42)` grouped by `client_hash_id`; no client overlap.

**Leakage check:** adding the label-derived `ctr_gap` feature intentionally produces AUC 1.0, confirming it must be excluded. The final feature set contains no label-derived or future-window feature.


## 4. Results (vs baseline)

| Method | Precision@50 | Precision@100 | ROC-AUC |
|---|---:|---:|---:|
| Rule baseline | 0.160 | 0.200 | 0.885 |
| Logistic regression | **0.480** | 0.520 | 0.799 |
| Random forest | 0.460 | **0.550** | 0.790 |
| Random/base-rate reference | 0.188 | 0.188 | 0.500 |

The learned models improve the top-of-queue metric that matches the operational decision. The baseline has higher ROC-AUC, so the result is not “the model is better at everything”; it is specifically better at the first 50–100 review positions.


In [ ]:
import pandas as pd

results = pd.DataFrame([
    ["Rule baseline", 0.160, 0.200, 0.885],
    ["Logistic regression", 0.480, 0.520, 0.799],
    ["Random forest", 0.460, 0.550, 0.790],
    ["Random/base-rate", 0.188, 0.188, 0.500],
], columns=["method","precision_at_50","precision_at_100","roc_auc"])
results


## 5. Limitations

1. The final supervised label is a same-window proxy, not a future intervention outcome.
2. The study is cross-sectional for one month and cannot establish causality.
3. The queue requires sufficient search exposure and does not cover new/unpublished pages.
4. Some high-impression, large-gap cases are under-scored and should be monitored.
5. Public reporting intentionally excludes client-level examples and private search information.


## 6. Ranked recommendations

1. Use the learned score to order a human review queue.
2. Verify tracking/attribution before assuming an extremely low CTR is a content problem.
3. Cap one client's share of a weekly queue at roughly 10%.
4. Manually inspect missing core fields before acting.
5. Keep title/meta edits human-approved; do not automate publishing.
6. Revalidate monthly with a grouped-client split; investigate if precision@50 falls below about 0.30.


## 7. Artifacts the paper embeds

The deployed page includes three public-safe charts:

- Precision@50 comparison
- Precision@100 comparison
- ROC-AUC comparison

The web page lives in `docs/index.html` and is intended for GitHub Pages deployment.


In [ ]:
# Reproducibility manifest for the public artifact.
manifest = {
    "random_state": 42,
    "primary_metric": "precision@K",
    "secondary_metric": "ROC-AUC",
    "split": "grouped by client_hash_id",
    "paper_path": "docs/index.html",
}
manifest


## Self-check

- [x] Question and decision are explicit.
- [x] Data release, tables, dates, exclusions, and scale are documented.
- [x] Methodology includes assumptions, features, label, baseline, validation, and leakage checks.
- [x] Results compare model and baseline on the same split.
- [x] Limitations and honest claim language are explicit.
- [x] Ranked recommendations are included.
- [x] Public artifact is free of client names, private URLs, raw queries, and credentials.


## Week 8 Showcase — 5-Minute Demo Outline

### 1. Question — ~45 seconds
Can a learned score prioritize search-visible content opportunities for a limited human review queue better than a transparent, position-aware CTR rule? This is the practical FlyRank content-review problem behind the lane.

### 2. Method — ~1 minute
- Aggregate daily search-performance data to a client-by-content decision grain.
- Apply the visibility threshold of at least 150 impressions.
- Define the March `ctr_gap` bottom-decile as a same-window screening proxy label.
- Compare Logistic Regression and Random Forest with a transparent rule baseline.
- Use a grouped client holdout so test clients do not overlap training clients.
- Run leakage checks and interpret the result as prioritization evidence, not causal evidence.

### 3. One Chart — ~1 minute
Show the **Precision@K model-vs-baseline chart**, focusing on Precision@50 and Precision@100 because the operational decision is which candidates should enter a small review queue.

### 4. One Honest Result — ~1 minute
On held-out clients, Logistic Regression achieved **0.480 Precision@50** and **0.520 Precision@100**, while Random Forest achieved **0.460** and **0.550**. The rule baseline achieved **0.160** and **0.200**. However, the rule baseline had the higher ROC-AUC (**0.885**), so top-of-queue usefulness and overall discrimination are not the same objective.

### 5. One Recommendation — ~1 minute
Use the learned score as a **human-review prioritization layer**: investigate high-priority, high-volume opportunities first, verify tracking and attribution, and retain the transparent rule baseline as a sanity check. Do not treat the score as proof that an intervention will increase CTR.


## Shareable Cuts

### Social post

During my ML internship, I worked on a large-scale search engagement problem using the FlyRank internship dataset. I built a position-aware CTR opportunity scoring workflow, compared Logistic Regression and Random Forest against a transparent rule baseline, and used client-grouped validation plus leakage checks to make the evaluation more realistic. The final result is designed as a human-review prioritization system: it improves top-of-queue precision on held-out clients, while the analysis deliberately avoids claiming that a content change will causally increase CTR.

### Employer-facing summary

I built a position-aware CTR opportunity scoring pipeline that ranks search-visible client-content observations for human review using Logistic Regression and Random Forest models. The project used FlyRank's large-scale pseudonymized search-performance dataset, a ≥150-impression visibility filter, and a client-grouped holdout with a transparent rule baseline. On held-out clients, the learned models substantially improved Precision@50/100 over the rule baseline, while the analysis showed why the output should be treated as a prioritization signal rather than causal evidence of future CTR improvement.
